In [1]:
import pandas as pd
from random import random

In [2]:
roles_df = pd.read_csv('framebank.csv')

In [3]:
def pair_differences_not_greater(lst, max_diff):
    lst = sorted(lst)
    for i in range(1, len(lst)):
        if abs(lst[i-1] - lst[i]) > max_diff:
            return False
    return True

In [4]:
def simple_kwic_output(kwics):
    if isinstance(kwics[0], list):
        return "\n".join(' '.join([tk.form for tk in kwic]).replace(' ,', ',').replace(' .', '.') for kwic in kwics)
    elif isinstance(kwics, str):
        return kwics
    else:
        return ' '.join([tk.form for tk in kwics])

In [5]:
class Token:
    def __init__(self, token_info):
        self.id, self.form, self.lemma, self.pos, self.xpos, self.gr, self.head, self.deprel, self.deps, self.misc = token_info.split('\t')
        self.id = int(self.id)
        if self.gr != '_':
            for cath_value in self.gr.split('|'):
                cathegory, value = cath_value.split('=')
                setattr(self, cathegory.lower(), value)
            
    def __str__(self):
        return '\t'.join(f'{k}:{v}' for k, v in self.__dict__.items() if not k.startswith('__') and not callable(k))
    
    def __repr__(self):
        return '\t'.join(f'{k}:{v}' for k, v in self.__dict__.items() if not k.startswith('__') and not callable(k))

In [6]:
class Sentence:
    def __init__(self, sentence_info):
        self.sent_id, self.sentence_text, *self.tokens = sentence_info.split('\n')
        self.sentence_text = self.sentence_text.replace('# text = ', '')
        self.tokens = [Token(token_info=token) for token in self.tokens]
    
    def __str__(self):
        return f'{self.sent_id}\n{self.sentence_text}\n'+'\n'.join(str(token) for token in self.tokens)
    
    def __repr__(self):
        return f'{self.sent_id}\n{self.sentence_text}\n'+'\n'.join(str(token) for token in self.tokens)
    
    def _search_by_token(self, query_token:str, kwic_len:int):
        if query_token in self.sentence_text:
            for tk in self.tokens:
                if tk.form == query_token:
                    query_token_index = tk.id
                    kwik = [tk for tk in self.tokens if abs(tk.id-query_token_index) <= kwic_len]
                    return kwik
        
    def _search_by_lemma(self, query_lemma:str, kwic_len:int):
        for tk in self.tokens:
            if tk.lemma == query_lemma:
                query_token_index = tk.id
                kwik = [tk for tk in self.tokens if abs(tk.id-query_token_index) <= kwic_len]
                return kwik
                
    def _general_search(self, kwic_len:int, token=None, lemma=None, pos=None, xpos=None, gr=None, deprel=None, **kwargs):
        query = {'form': token, 'lemma':lemma, 'pos': pos, 'xpos': xpos, 'gr': gr, 'deprel': deprel}
        query = {k:v for k, v in query.items() if v is not None}
        if kwargs:
            for k, v in kwargs.items():
                query[k] = v
        
        for tk in self.tokens:
            condition = all(tk.__getattribute__(key)==value for key, value in query.items())
            if condition:
                query_token_index = tk.id
                kwik = [tk for tk in self.tokens if abs(tk.id-query_token_index) <= kwic_len]
                return kwik
    
    def _multiword_search(self, token_descriptions, kwik_len=1, allow_distance=1): # token_descriptions is list of dicts that describe each token with some features
        all_tokens_present = all(
            any(
                all(tk.__getattribute__(key)==value for key, value in tk_descripion.items()) 
                    for tk in self.tokens
                )
                for tk_descripion in token_descriptions
        )
        if all_tokens_present:
            token_indexes = []
            for token_description in token_descriptions:
                for tk in self.tokens:
                    condition = all(tk.__getattribute__(key)==value for key, value in token_description.items())
                    if condition:
                        token_indexes.append(tk.id)
            if pair_differences_not_greater(token_indexes, allow_distance):
                kwik = [tk for tk in self.tokens 
                        if (
                            abs(min(token_indexes) - tk.id) <= kwik_len 
                            or abs(max(token_indexes) - tk.id) <= kwik_len
                            or tk.id >=min(token_indexes) and tk.id <= max(token_indexes)
                            )
                        ]
                return kwik

In [7]:
class Corpus:
    def __init__(self):
        self.sentences = []

    def load_from_file(self, filepath):
        with open(filepath, encoding='utf-8') as corpus_file:
            self.sentences = [Sentence(sent) for sent in corpus_file.read().split('\n\n') if sent]

    def search_by_token(self, token, n_examples=5, kwic_len=5):
        qwery_answer = []
        for sentence in sorted(self.sentences, key=lambda x: random()):
            if len(qwery_answer) == n_examples:
                return qwery_answer
            c = sentence._search_by_token(query_token=token, kwic_len=kwic_len)
            if c:
                qwery_answer.append(c)
        if qwery_answer:
            return qwery_answer
        return f'Примеров для {token=} в корпусе не нашлось.'
    
    def search_by_lemma(self, lemma, n_examples=5, kwic_len=5):
        qwery_answer = []
        for sentence in sorted(self.sentences, key=lambda x: random()):
            if len(qwery_answer) == n_examples:
                return qwery_answer
            c = sentence._search_by_lemma(query_lemma=lemma, kwic_len=kwic_len)
            if c:
                qwery_answer.append(c)
        if qwery_answer:
            return qwery_answer
        return f'Примеров для {lemma=} в корпусе не нашлось.'

    def general_search(self, token=None, lemma=None, pos=None, xpos=None, gr=None, deprel=None, n_examples=5, kwic_len=5, **kwargs):
        qwery_answer = []
        for sentence in sorted(self.sentences, key=lambda x: random()):
            if len(qwery_answer) == n_examples:
                return qwery_answer
            c = sentence._general_search(token=token, lemma=lemma, pos=pos, xpos=xpos, gr=gr, deprel=deprel, kwic_len=kwic_len, **kwargs)
            if c:
                qwery_answer.append(c)
        if qwery_answer:
            return qwery_answer
        plug = ' '.join([f'{x=}' for x in [lemma, pos, xpos, gr, deprel] if x is not None])
        return f'Примеров для {plug} в корпусе не нашлось.'
    
    def search_multiword(self, token_descriptions, kwik_len=1, allow_distance=1, n_examples=5, ):
        qwery_answer = []
        for sentence in sorted(self.sentences, key=lambda x: random()):
            if len(qwery_answer) == n_examples:
                return qwery_answer
            c = sentence._multiword_search(token_descriptions, kwik_len, allow_distance)
            if c:
                qwery_answer.append(c)
        if qwery_answer:
            return qwery_answer
        return 'Поиск не дал результатов'
    
    def search_by_role(self, keyword_lemma='', role='', kwik_len=1, allow_distance=1, n_examples=5):
        if keyword_lemma and role:
            examples = roles_df[(roles_df.KeyLexemes==keyword_lemma) & (roles_df.Role==role)]
        elif keyword_lemma:
            examples = roles_df[roles_df.KeyLexemes==keyword_lemma]
        elif role:
            examples = roles_df[roles_df.Role==role]
        else:
            return 'Задан пустой запрос'
        output = [] 
        for example in examples.values:
            collocate = example[0]
            keyword_lemma = keyword_lemma if keyword_lemma else example[4]
            
            if collocate.isalpha():
                response = simple_kwic_output(self.search_multiword([{'lemma': keyword_lemma}, {'form': collocate}], kwik_len, allow_distance, n_examples))
                if response != 'Поиск не дал результатов':
                    output.append([example[2], collocate, response])
            
            elif all(l.isalpha() or l.isspace() for l in collocate):
                query = [{'lemma': keyword_lemma}] + [{'form': part} for part in collocate.split()]
                response = simple_kwic_output(self.search_multiword(query, kwik_len, allow_distance, n_examples))
                if response != 'Поиск не дал результатов':
                    output.append([example[2], collocate, response])
            
            else:
                print('ERROR COLLOCATE', collocate, 'LEMMA', keyword_lemma)
        return output

In [8]:
CORPUS = Corpus()
CORPUS.load_from_file('NPlus1.txt')

In [9]:
print(simple_kwic_output(CORPUS.general_search(lemma='видеть', kwic_len=10, n_examples=10)))

Качество изображений непозволило детально его разглядеть, однако ученым удалось увидеть, где лежит космический аппарат, парашют, который должен
Тогда удалось увидеть, как в фазе глубокого сна клетки мозга слегка "
Те же, кто видел отрицательные комментарии, наоборот, в большинстве своем оценивали кандидата
помощью простой модели ученые выяснили, что объектам достаточно просто видеть и реагировать на моментальное присутствие других объектов.
либо высвобождали препарат слишком быстро, либо непозволяли пациенту нормально видеть.
даже в совершенно случайном изображении -- например, облаков -- увидит какие-то его фрагменты.
их бега. Дело в том, что некоторые специалисты не видят для ящериц особой пользы от двуногого передвижения, полагая,
Антигелий - 3 увидели в 1971 году.
Предполагалось, что радар электромобиля полуприцеп " не увидел ", поскольку его угол возвышения был специально занижен,
Вы можете в точности видеть, как ведут себя отдельные клетки во время процесса регенерации


In [29]:
print(simple_kwic_output(CORPUS.search_multiword([{'lemma': 'видеть'}, {'lemma': 'глаз'}], kwik_len=10, n_examples=10, allow_distance=2)))

располагают еще один светоделитель, пропускающий лишь половину фотонов. Напрямую увидеть глазом единичные фотоны невозможно -- светочувствительные клетки сетчатки требуютпо меньшей мере
оптического диапазона ( 400 - 800 нанометров ), наш глаз способен видеть и инфракрасное излучение ( около 1000 нанометров ), благодаря
В своих дальнейших экспериментах Холмс надеется " увидеть " глазами подопытных суперпозицию фотонов. Оптические возможности сетчатки до сих пор являются


In [31]:
print(simple_kwic_output(CORPUS.search_multiword([{'lemma': 'видеть'}, {'lemma': 'мочь'}], kwik_len=10, n_examples=10, allow_distance=1)))

что речь идет именно о тех галактиках, которые мы можем увидеть, а не о всех существующих в настоящий момент (
Например, птицы не могут видеть, что происходит за спиной, а рыбы способны получать
ночной режим, при котором летчики с помощью тепловизионных камер могут видеть не только световую индикацию, но очертания кораблей. Такой
Все животные могли видеть ислышать друг друга.
прозрачном щитке шлема командира боевой машины, благодаря чему он может видеть, что происходит внутри машины.
только 536 людей из семи миллиардов побывали в космосе и смогли увидеть наш мир совсем по-другому, поэтому цель проекта -- дать
Вы действительно можете видеть необходимые закономерности при такой визуализации данных ",-- сказал руководитель
Отделив реакторный вклад, исследователи смогли увидеть в 2010 году примерно 10 геонейтрино.
смещают картинку на нашлемном дисплее. Благодаря системе кругового обзора летчик может увидеть, что происходит, например, под или за самолетом
Международная группа астрономов под

In [10]:
print(simple_kwic_output(CORPUS.search_multiword([{'lemma': 'видеть'}, {'pos': 'NOUN'}], kwik_len=10, n_examples=10, allow_distance=2)))

Говоря более современным языком, жители страны увидели взрыв новой или сверхновой.
" Было изумительно увидеть эти особенности так отчетливо.
Тогда ученые увидели ватмосфере карликовой планеты присутствие небольшого количества водяного пара.
Последний раз представителя вида видели профессиональные рыбаки в2009году.
" Вовсех комментариях люди выражают желание увидеть геймплей.
Благодаря этому боец увидит объемное изображение.
Наблюдая рост цветных клеток, авторы исследования смогли увидеть процесс регенерации тканей.
Ввысоком разрешении ееможно увидеть здесь.
Процесс создания комнаты можно увидеть ниже :
Врезультате они смогли увидеть вВеликом аттракторе 883галактики, треть которых не наблюдалась ранее.


In [35]:
CORPUS.search_by_role(keyword_lemma='мочь')

ERROR COLLOCATE всё-таки LEMMA мочь


[['отрицание',
  'не',
  'которую не могут "\nполя не могут быть\nспециалисты не могут вести\nзамка может не только\nкоторые не смогли разбить'],
 ['самостоятельность',
  'сам',
  'пользователь может сам обвести\nпользователь сам может выбрать\nи сам может быть\nробот сам может чертить\nне может сам по'],
 ['отрицание',
  'не',
  'полости не могут обеспечить\nмикроскопы не могут "\nпрограммы не могут играть\nкоторую не могут "\nбольше не сможете собираться'],
 ['время',
  'теперь',
  'компании теперь могут получить\nученые теперь могут применить\nгибрид теперь сможет отвечать\nмы можем теперь найти'],
 ['условное наклонение',
  'бы',
  'что могло бы помочь датировать\nбедствий могли бы выполнять\nпоследствиям могло бы привести\nне могла бы удержать\nкоторые могли бы привести'],
 ['косв.',
  'ли',
  '" может ли компьютер\n, могут ли планеты\n, могло ли это\n, может ли человек\n, могут ли нетренированные'],
 ['узуальность',
  'не всегда',
  'компьютеры не всегда смогут обеспечивать\nмага

In [17]:
CORPUS.search_by_role(keyword_lemma='видеть', role='оценка')

[['оценка', 'хорошо', 'был хорошо видим для\nочень хорошо видеть в']]

In [18]:
CORPUS.search_by_role(keyword_lemma='сказать', role='отрицание')

[['отрицание', 'не', 'точно сказать не могут']]

In [36]:
roles_df[roles_df.KeyLexemes=='мочь']

,Phrase,Form,Role,Type,KeyLexemes
11797,не хотелось огорчать якова мироновича,CL,мотивировка,Circum,мочь
13016,вот только,PART,дискурс,Modal,мочь
13018,не,PART,отрицание,Modal,мочь
13020,как это,ADVPRO,мотивировка,Circum,мочь
13022,сам,APRO,самостоятельность,Circum,мочь
13024,не,PART,отрицание,NaN,мочь
13026,тем более,ADV,дискурс,Circum,мочь
13028,ещё,ADV,NaN,Circum,мочь
13032,теперь,ADV,время,Circum,мочь
13036,по натуре,по + Sdat,причина,Circum,мочь
